In [1]:
import pandas
import requests
from collections import Counter
from pathlib import Path

In [2]:
server = "http://localhost:8000"

In [3]:
def get_all_pages(url):
    results = []
    while url is not None:
        resp = requests.get(url, headers={"Accept": "application/json"})
        data = resp.json()
        url = data.get("next")
        results.extend(data.get("results", []))
        print("Retrieved {} of {}".format(len(files), data.get("count")))
        
    return results


In [4]:
def get_object(url, cache={}):
    if not isinstance(url, str):
        return url

    if url not in cache:
        resp = requests.get(url)
        if resp.status_code == 200:
            cache[url] = resp.json()
            
    return cache[url]


In [5]:
def get_files(server):
    url = "{server}/subpool-in-run-file".format(server=server)

    files = []
    while url is not None:
        resp = requests.get(url, headers={"Accept": "application/json"})
        data = resp.json()
        url = data.get("next")
        for f in data.get("results", []):
            f["sequencing_run"] = get_object(f["sequencing_run"])
            f["subpool_run"] = get_object(f["subpool_run"])
            f["subpool_run"]["subpool"] = get_object(f["subpool_run"]["subpool"])
            files.append(f)
        print("Retrieved {} of {}".format(len(files), data.get("count")))
        
    return files


In [6]:
files = get_files(server)
len(files)

Retrieved 100 of 3119
Retrieved 200 of 3119
Retrieved 300 of 3119
Retrieved 400 of 3119
Retrieved 500 of 3119
Retrieved 600 of 3119
Retrieved 700 of 3119
Retrieved 800 of 3119
Retrieved 900 of 3119
Retrieved 1000 of 3119
Retrieved 1100 of 3119
Retrieved 1200 of 3119
Retrieved 1300 of 3119
Retrieved 1400 of 3119
Retrieved 1500 of 3119
Retrieved 1600 of 3119
Retrieved 1700 of 3119
Retrieved 1800 of 3119
Retrieved 1900 of 3119
Retrieved 2000 of 3119
Retrieved 2100 of 3119
Retrieved 2200 of 3119
Retrieved 2300 of 3119
Retrieved 2400 of 3119
Retrieved 2500 of 3119
Retrieved 2600 of 3119
Retrieved 2700 of 3119
Retrieved 2800 of 3119
Retrieved 2900 of 3119
Retrieved 3000 of 3119
Retrieved 3100 of 3119
Retrieved 3119 of 3119


3119

In [7]:
files[0]

{'@id': 'http://localhost:8000/subpool-in-run-file/1/',
 'md5sum': '57e977fdfe32850db83ee5f7addb7165',
 'filename': 'igvf_003/nextseq2/003_13A_R1.fastq.gz',
 'flowcell_id': 'AAC3V7HHV',
 'lane': None,
 'read': 'R1',
 'sequencing_run': {'@id': 'http://localhost:8000/sequencing-run/1/',
  'name': 'igvf_003/nextseq2',
  'run_date': '2023-10-02',
  'stranded': 'R',
  'platform': {'@id': 'http://localhost:8000/platform/nextseq2000/',
   'igvf_id': '/platform-terms/EFO_0010963/',
   'display_name': 'Nextseq 2000',
   'family': 'illumina'},
  'plate': {'@id': 'http://localhost:8000/split-seq-plate/IGVF_003/',
   'name': 'IGVF_003',
   'size': '96',
   'pool_location': None,
   'date_performed': None,
   'barcoded_cell_counter': None,
   'volume_of_nuclei': None,
   'sequencing_runs': [{'@id': 'http://localhost:8000/sequencing-run/1/',
     'name': 'igvf_003/nextseq2',
     'run_date': '2023-10-02',
     'stranded': 'R',
     'platform': {'@id': 'http://localhost:8000/platform/nextseq2000/',
 

From https://docs.google.com/presentation/d/1W7lhTxsZH5tOY095HTo2353wUMfQLuoGHyD5V6CSMWY/edit#slide=id.g22b43c49617_0_9

```
${plate}_${subpool}_${protocol}_${platform}_$N{run}_R${read}


plate: igvf003, igvf004, etc.
subpool: 2 digit ID # for pool, i.e. 01, 02, 03….16. 
platform: next, nova, grid, prom
run: typically 1/2 for nova (1 for 008/008b), typically 1 for next (2 for bridge, 2 for exome capture subpools), many for grid, maybe 1 for prom
protocol: D or E (D = default, E = exome capture)
read: 1 for Nanopore (grid & prom) and 1/2 for Illumina (next & nova)
```

In [8]:
def format_barcode(value):
    assert value is not None
    if isinstance(value, int) or value.isdigit():
        return "B{:02}".format(int(value))
    else:
        return "B{}".format(value)

assert format_barcode(13) == "B13"
assert format_barcode("3") == "B03"
assert format_barcode("UDI03") == "BUDI03"

In [9]:
short_platform_names = {
    "/platform-terms/EFO_0010963/": "next",
    "/platform-terms/EFO_0008637/": "nova",
    "/platform-terms/EFO_0008632/": "mini",
    "/platform-terms/EFO_0008634/": "prom",

}

short_protocol_names = {
    "NO": "D",
    "EX": "E",
}

def decode_run_name(run_name):
    plate, run = run_name.split("/")
    
    return {
        "next": 1,
        "next1": 1,
        "next2": 2,
        "nextseq": 1,
        "nextseq1": 1,
        "nextseq2": 2,
        "nova1": 1,
        "nova2": 2,
    }[run]

records = []
for f in files:
    assert f["filename"].endswith(".fastq.gz")
    file_type = ".fastq.gz"
    
    if f["sequencing_run"]["platform"]["family"] == "nanopore":
        # Skip nanopore
        continue

    plate = f["sequencing_run"]["plate"]["name"].lower()
    subpool_index = format_barcode(f["subpool_run"]["subpool"]["index"])
    platform = short_platform_names[f["sequencing_run"]["platform"]["igvf_id"]]
    dirname = f["sequencing_run"]["name"]
    run = decode_run_name(f["sequencing_run"]["name"])
    protocol = short_protocol_names[f["subpool_run"]["subpool"]["selection_type"]]
    if f.get("lane") is not None:
        lane = "_L{:03}".format(f.get("lane"))
    else:
        lane = ""
    read = "_{}".format(f.get("read"))
    target_name = f"{dirname}/{plate}_{subpool_index}_{protocol}_{platform}_N{run}{lane}{read}{file_type}"
    accessions = [x["see_also"] for x in f["accession"] if x["name"].startswith("IGVF")]
    if len(accessions) == 0:
        accession = ""
    elif len(accessions) == 1:
        accession = accessions[0]
    else:
        raise RuntimeError(f"Itegrity error {f}")
    records.append({
        "source": f["filename"],
        "target": target_name,
        "md5sum": f["md5sum"],
        "accession": accession,
    })

records = pandas.DataFrame(records).sort_values("source")
records

,source,target,md5sum,accession
2,igvf_003/nextseq/003_67A_R1.fastq.gz,igvf_003/nextseq/igvf_003_B02_D_next_N1_R1.fas...,35350d53cf858267d294bce69e475c83,https://api.data.igvf.org/sequence-files/IGVFF...
3,igvf_003/nextseq/003_67A_R2.fastq.gz,igvf_003/nextseq/igvf_003_B02_D_next_N1_R2.fas...,00c68ff0f9e0217d422c57e8948d4bb4,https://api.data.igvf.org/sequence-files/IGVFF...
4,igvf_003/nextseq/003_67B_R1.fastq.gz,igvf_003/nextseq/igvf_003_B03_D_next_N1_R1.fas...,79db765143bb23e14c613c9e1339e9f7,https://api.data.igvf.org/sequence-files/IGVFF...
5,igvf_003/nextseq/003_67B_R2.fastq.gz,igvf_003/nextseq/igvf_003_B03_D_next_N1_R2.fas...,c4787058f6b8304b1ddae0a55b48fb13,https://api.data.igvf.org/sequence-files/IGVFF...
6,igvf_003/nextseq/003_8A_R1.fastq.gz,igvf_003/nextseq/igvf_003_B01_D_next_N1_R1.fas...,bb696e63b6a8710ec8435b60ef6c81f8,
...,...,...,...,...
3098,igvf_b01/next2/B01_13F_R2.fastq.gz,igvf_b01/next2/igvf_b01_B06_D_next_N2_R2.fastq.gz,5f49893f54aaf88d06ca5834268a4e55,https://api.data.igvf.org/sequence-files/IGVFF...
3100,igvf_b01/next2/B01_13G_R1.fastq.gz,igvf_b01/next2/igvf_b01_B07_E_next_N2_R1.fastq.gz,2d4a0beb2e31d5981ebc56748297395b,https://api.data.igvf.org/sequence-files/IGVFF...
3102,igvf_b01/next2/B01_13G_R2.fastq.gz,igvf_b01/next2/igvf_b01_B07_E_next_N2_R2.fastq.gz,b3f5e2b91fc926c070c85c40e654cb0d,https://api.data.igvf.org/sequence-files/IGVFF...
3104,igvf_b01/next2/B01_13H_R1.fastq.gz,igvf_b01/next2/igvf_b01_B08_D_next_N2_R1.fastq.gz,cef06002f2986f0c305da0031910c1f0,https://api.data.igvf.org/sequence-files/IGVFF...


As of Sept 5th, there's ~2931 fastq files.

In [10]:
with open("great-rename.txt", "wt") as outstream:
    full_collisions = Counter()
    name_collisions = Counter()
    for i, row in records.iterrows():
        full_collisions[row["target"]] += 1
        name_collisions[Path(row["target"]).name] += 1
        record = "mv -n\t{}\t{}\t\t#{}\t{}".format(row["source"], row["target"], row["md5sum"], row["accession"])
        print(record)
        outstream.write(record)
        outstream.write("\n")

mv -n	igvf_003/nextseq/003_67A_R1.fastq.gz	igvf_003/nextseq/igvf_003_B02_D_next_N1_R1.fastq.gz		#35350d53cf858267d294bce69e475c83	https://api.data.igvf.org/sequence-files/IGVFFI7878WYCE/
mv -n	igvf_003/nextseq/003_67A_R2.fastq.gz	igvf_003/nextseq/igvf_003_B02_D_next_N1_R2.fastq.gz		#00c68ff0f9e0217d422c57e8948d4bb4	https://api.data.igvf.org/sequence-files/IGVFFI6614EZDQ/
mv -n	igvf_003/nextseq/003_67B_R1.fastq.gz	igvf_003/nextseq/igvf_003_B03_D_next_N1_R1.fastq.gz		#79db765143bb23e14c613c9e1339e9f7	https://api.data.igvf.org/sequence-files/IGVFFI5879FSBN/
mv -n	igvf_003/nextseq/003_67B_R2.fastq.gz	igvf_003/nextseq/igvf_003_B03_D_next_N1_R2.fastq.gz		#c4787058f6b8304b1ddae0a55b48fb13	https://api.data.igvf.org/sequence-files/IGVFFI5078LLAB/
mv -n	igvf_003/nextseq/003_8A_R1.fastq.gz	igvf_003/nextseq/igvf_003_B01_D_next_N1_R1.fastq.gz		#bb696e63b6a8710ec8435b60ef6c81f8	
mv -n	igvf_003/nextseq/003_8A_R2.fastq.gz	igvf_003/nextseq/igvf_003_B01_D_next_N1_R2.fastq.gz		#21967ba1a5a3218508961fac48

mv -n	igvf_004/nova1/Sublibrary_12_S11_L001_R1_001.fastq.gz	igvf_004/nova1/igvf_004_B12_D_nova_N1_L001_R1.fastq.gz		#a355e2eda77543b1446c55569e84f082	https://api.data.igvf.org/sequence-files/IGVFFI2463AKSF/
mv -n	igvf_004/nova1/Sublibrary_12_S11_L001_R2_001.fastq.gz	igvf_004/nova1/igvf_004_B12_D_nova_N1_L001_R2.fastq.gz		#dc80f2b4b2a4eaaeec93528edd97e902	https://api.data.igvf.org/sequence-files/IGVFFI2842PLJB/
mv -n	igvf_004/nova1/Sublibrary_12_S11_L002_I1_001.fastq.gz	igvf_004/nova1/igvf_004_B12_D_nova_N1_L002_I1.fastq.gz		#0c2b06780e9524591359bbb4558e33e1	
mv -n	igvf_004/nova1/Sublibrary_12_S11_L002_R1_001.fastq.gz	igvf_004/nova1/igvf_004_B12_D_nova_N1_L002_R1.fastq.gz		#1bdd813ee656c008a1e67e460ca3bf48	https://api.data.igvf.org/sequence-files/IGVFFI5105WGDO/
mv -n	igvf_004/nova1/Sublibrary_12_S11_L002_R2_001.fastq.gz	igvf_004/nova1/igvf_004_B12_D_nova_N1_L002_R2.fastq.gz		#833f7751cccce848423a966040c962c7	https://api.data.igvf.org/sequence-files/IGVFFI4752XESZ/
mv -n	igvf_004/nova1/

mv -n	igvf_005/nova1/Sublibrary_6_S5_L004_I1_001.fastq.gz	igvf_005/nova1/igvf_005_B06_D_nova_N1_L004_I1.fastq.gz		#1aeb39214f7dd9d7c9250f5f01882593	
mv -n	igvf_005/nova1/Sublibrary_6_S5_L004_R1_001.fastq.gz	igvf_005/nova1/igvf_005_B06_D_nova_N1_L004_R1.fastq.gz		#8f947dcc0ed0bafda54343ba697bae78	https://api.data.igvf.org/sequence-files/IGVFFI7976ERLA/
mv -n	igvf_005/nova1/Sublibrary_6_S5_L004_R2_001.fastq.gz	igvf_005/nova1/igvf_005_B06_D_nova_N1_L004_R2.fastq.gz		#fa5cc54d55647c4ecb10526f43339f96	https://api.data.igvf.org/sequence-files/IGVFFI1404RRLZ/
mv -n	igvf_005/nova1/Sublibrary_7_S6_L001_I1_001.fastq.gz	igvf_005/nova1/igvf_005_B07_D_nova_N1_L001_I1.fastq.gz		#9e7879872cc7148e03b55ba694b76d3f	
mv -n	igvf_005/nova1/Sublibrary_7_S6_L001_R1_001.fastq.gz	igvf_005/nova1/igvf_005_B07_D_nova_N1_L001_R1.fastq.gz		#2141ff31d3124d55f43361e0bf541430	https://api.data.igvf.org/sequence-files/IGVFFI6749XMCZ/
mv -n	igvf_005/nova1/Sublibrary_7_S6_L001_R2_001.fastq.gz	igvf_005/nova1/igvf_005_B07_D

mv -n	igvf_005/nova2/Sublibrary_11_S10_L002_I1_001.fastq.gz	igvf_005/nova2/igvf_005_B11_D_nova_N2_L002_I1.fastq.gz		#209d1b8133fad1f9ae8b4175b8d51caa	
mv -n	igvf_005/nova2/Sublibrary_11_S10_L002_R1_001.fastq.gz	igvf_005/nova2/igvf_005_B11_D_nova_N2_L002_R1.fastq.gz		#9ed7561106f42d0af5efbee69c1da8b0	https://api.data.igvf.org/sequence-files/IGVFFI1380GMCV/
mv -n	igvf_005/nova2/Sublibrary_11_S10_L002_R2_001.fastq.gz	igvf_005/nova2/igvf_005_B11_D_nova_N2_L002_R2.fastq.gz		#21e611873c273154e14d4cf8c7143ca3	https://api.data.igvf.org/sequence-files/IGVFFI0390CQUL/
mv -n	igvf_005/nova2/Sublibrary_11_S10_L003_I1_001.fastq.gz	igvf_005/nova2/igvf_005_B11_D_nova_N2_L003_I1.fastq.gz		#d231b8c84cf5e3ddede32ce828a52e93	
mv -n	igvf_005/nova2/Sublibrary_11_S10_L003_R1_001.fastq.gz	igvf_005/nova2/igvf_005_B11_D_nova_N2_L003_R1.fastq.gz		#5edcccf9da8978669c049576db7d4016	https://api.data.igvf.org/sequence-files/IGVFFI3203LXON/
mv -n	igvf_005/nova2/Sublibrary_11_S10_L003_R2_001.fastq.gz	igvf_005/nova2/ig

mv -n	igvf_007/nova1/Sublibrary_14_S13_L003_I1_001.fastq.gz	igvf_007/nova1/igvf_007_B14_D_nova_N1_L003_I1.fastq.gz		#e155fb3cd6d9de4f5528bd18baac80cb	
mv -n	igvf_007/nova1/Sublibrary_14_S13_L003_R1_001.fastq.gz	igvf_007/nova1/igvf_007_B14_D_nova_N1_L003_R1.fastq.gz		#d51eedac57c9837d388864cb35b8af55	https://api.data.igvf.org/sequence-files/IGVFFI9368ZOAM/
mv -n	igvf_007/nova1/Sublibrary_14_S13_L003_R2_001.fastq.gz	igvf_007/nova1/igvf_007_B14_D_nova_N1_L003_R2.fastq.gz		#5b947ccd5e1671507dd40abc81a3a61e	https://api.data.igvf.org/sequence-files/IGVFFI5410EZVQ/
mv -n	igvf_007/nova1/Sublibrary_14_S13_L004_I1_001.fastq.gz	igvf_007/nova1/igvf_007_B14_D_nova_N1_L004_I1.fastq.gz		#00a3cb3e6be1c4988f831f61d670e272	
mv -n	igvf_007/nova1/Sublibrary_14_S13_L004_R1_001.fastq.gz	igvf_007/nova1/igvf_007_B14_D_nova_N1_L004_R1.fastq.gz		#beb4a87a1279c6911a0a5476f3760134	https://api.data.igvf.org/sequence-files/IGVFFI8801VAJL/
mv -n	igvf_007/nova1/Sublibrary_14_S13_L004_R2_001.fastq.gz	igvf_007/nova1/ig

...

mv -n	igvf_010/nextseq2/010_13A_R1.fastq.gz	igvf_010/nextseq2/igvf_010_B07_E_next_N2_R1.fastq.gz		#ab6a95e43234701e4feac682e627517c	https://api.data.igvf.org/sequence-files/IGVFFI2235MCOP/
mv -n	igvf_010/nextseq2/010_13A_R2.fastq.gz	igvf_010/nextseq2/igvf_010_B07_E_next_N2_R2.fastq.gz		#46dfe5ce2abb19863c63ea3855bbbb31	https://api.data.igvf.org/sequence-files/IGVFFI3650UECM/
mv -n	igvf_010/nova1/Sublibrary_10_S9_L001_I1_001.fastq.gz	igvf_010/nova1/igvf_010_B10_D_nova_N1_L001_I1.fastq.gz		#4d193266ad0ccfde8a75085a47d40800	
mv -n	igvf_010/nova1/Sublibrary_10_S9_L001_R1_001.fastq.gz	igvf_010/nova1/igvf_010_B10_D_nova_N1_L001_R1.fastq.gz		#20b096fc703b2b90d8113034dae96ca5	https://api.data.igvf.org/sequence-files/IGVFFI8471EXAF/
mv -n	igvf_010/nova1/Sublibrary_10_S9_L001_R2_001.fastq.gz	igvf_010/nova1/igvf_010_B10_D_nova_N1_L001_R2.fastq.gz		#bf2ae866b88f1732773141ae4af1c015	https://api.data.igvf.org/sequence-files/IGVFFI4453WDKD/
mv -n	igvf_010/nova1/Sublibrary_10_S9_L002_I1_001.fastq.gz	i

mv -n	igvf_010/nova2/Sublibrary_9_S8_L003_R2_001.fastq.gz	igvf_010/nova2/igvf_010_B09_D_nova_N2_L003_R2.fastq.gz		#aabf438de20c0591423159e9145f3ed8	https://api.data.igvf.org/sequence-files/IGVFFI3567GIOG/
mv -n	igvf_010/nova2/Sublibrary_9_S8_L004_I1_001.fastq.gz	igvf_010/nova2/igvf_010_B09_D_nova_N2_L004_I1.fastq.gz		#56d1e55245bd351403940c23f81b5d20	
mv -n	igvf_010/nova2/Sublibrary_9_S8_L004_R1_001.fastq.gz	igvf_010/nova2/igvf_010_B09_D_nova_N2_L004_R1.fastq.gz		#bde6062e6dc17d346d1c276f93383de9	https://api.data.igvf.org/sequence-files/IGVFFI6447ZISS/
mv -n	igvf_010/nova2/Sublibrary_9_S8_L004_R2_001.fastq.gz	igvf_010/nova2/igvf_010_B09_D_nova_N2_L004_R2.fastq.gz		#09f27cf379d08d9d98ac96b6e4d5cb46	https://api.data.igvf.org/sequence-files/IGVFFI6280LIFM/
mv -n	igvf_011/nextseq/011_67M_R1.fastq.gz	igvf_011/nextseq/igvf_011_B14_D_next_N1_R1.fastq.gz		#1a42061f3fbac12749718ba52148a8b7	https://api.data.igvf.org/sequence-files/IGVFFI0091JSEG/
mv -n	igvf_011/nextseq/011_67M_R2.fastq.gz	igvf_0

mv -n	igvf_011/nova1/Sublibrary_14_S13_L001_I1_001.fastq.gz	igvf_011/nova1/igvf_011_B14_D_nova_N1_L001_I1.fastq.gz		#056f159e153bd0d8ec019de48c11c227	
mv -n	igvf_011/nova1/Sublibrary_14_S13_L001_R1_001.fastq.gz	igvf_011/nova1/igvf_011_B14_D_nova_N1_L001_R1.fastq.gz		#7d41cbb379750bfb303cc4672a6797fc	https://api.data.igvf.org/sequence-files/IGVFFI1629IPLY/
mv -n	igvf_011/nova1/Sublibrary_14_S13_L001_R2_001.fastq.gz	igvf_011/nova1/igvf_011_B14_D_nova_N1_L001_R2.fastq.gz		#91ae2e7dce791bdf6e6dee044271b72e	https://api.data.igvf.org/sequence-files/IGVFFI6018WMYW/
mv -n	igvf_011/nova1/Sublibrary_14_S13_L002_I1_001.fastq.gz	igvf_011/nova1/igvf_011_B14_D_nova_N1_L002_I1.fastq.gz		#d6ab071cb0a6c08367771be854742a1f	
mv -n	igvf_011/nova1/Sublibrary_14_S13_L002_R1_001.fastq.gz	igvf_011/nova1/igvf_011_B14_D_nova_N1_L002_R1.fastq.gz		#6f0a9dd863d593d0b04ade91de7322fd	https://api.data.igvf.org/sequence-files/IGVFFI6437VRFR/
mv -n	igvf_011/nova1/Sublibrary_14_S13_L002_R2_001.fastq.gz	igvf_011/nova1/ig

mv -n	igvf_013/nova1/Sublibrary_11_S10_L001_R1_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI11_D_nova_N1_L001_R1.fastq.gz		#fbe180e344b12170276455fe0c379f05	https://api.data.igvf.org/sequence-files/IGVFFI3393MRMQ/
mv -n	igvf_013/nova1/Sublibrary_11_S10_L001_R2_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI11_D_nova_N1_L001_R2.fastq.gz		#c217359da67ed1e4aa47fb6f7dfb938d	https://api.data.igvf.org/sequence-files/IGVFFI5888IJXV/
mv -n	igvf_013/nova1/Sublibrary_11_S10_L002_I1_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI11_D_nova_N1_L002_I1.fastq.gz		#62e54c54f12761c9a4eb55efd4634786	
mv -n	igvf_013/nova1/Sublibrary_11_S10_L002_I2_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI11_D_nova_N1_L002_I2.fastq.gz		#28a3e5d54a5a3abb9474044087d6f8e6	https://api.data.igvf.org/sequence-files/IGVFFI3645MBBE/
mv -n	igvf_013/nova1/Sublibrary_11_S10_L002_R1_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI11_D_nova_N1_L002_R1.fastq.gz		#64d8f880781051e79d492749028df9d6	https://api.data.igvf.org/sequence-files/IGVFFI3386XZVC/
mv -n	

mv -n	igvf_013/nova1/Sublibrary_14_S13_L001_R1_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI14_D_nova_N1_L001_R1.fastq.gz		#95b31852af7b1f095051ec54ccda6077	https://api.data.igvf.org/sequence-files/IGVFFI6413FURD/
mv -n	igvf_013/nova1/Sublibrary_14_S13_L001_R2_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI14_D_nova_N1_L001_R2.fastq.gz		#6291efed7d81296298c580297ccd3d20	https://api.data.igvf.org/sequence-files/IGVFFI3740TMNI/
mv -n	igvf_013/nova1/Sublibrary_14_S13_L002_I1_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI14_D_nova_N1_L002_I1.fastq.gz		#4b6a2d41ecf5c19082ce2f91b646d71a	
mv -n	igvf_013/nova1/Sublibrary_14_S13_L002_I2_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI14_D_nova_N1_L002_I2.fastq.gz		#2c6502375d585b94b25d309689baba05	https://api.data.igvf.org/sequence-files/IGVFFI5266OMEM/
mv -n	igvf_013/nova1/Sublibrary_14_S13_L002_R1_001.fastq.gz	igvf_013/nova1/igvf_013_BUDI14_D_nova_N1_L002_R1.fastq.gz		#ce32e6bce65f03bc15ebbf9d01c2a5ff	https://api.data.igvf.org/sequence-files/IGVFFI7784OEKM/
mv -n	

In [11]:
%debug

ERROR:root:No traceback has been produced, nothing to debug.


In [12]:
for k in full_collisions:
    count = full_collisions[k]
    if count > 1:
        print(k, count)

In [13]:
for k in name_collisions:
    count = name_collisions[k]
    if count > 1:
        print(k, count)

In [14]:
# 0dbdc8a7fe445f8a9e6bbbd7a7d42cfd nanopore igvf003_8A_lig-ss_7.fastq.gz 
# dc68518d2246ead1d5931ac70db8e1c5 illumina fastq 008B_13A_R1

In [15]:
for f in files:
    if f["md5sum"] == "dc68518d2246ead1d5931ac70db8e1c5":
        break
#f

In [16]:
md5_collisions = {}
for f in files:
    md5_collisions.setdefault(f["md5sum"], []).append(f)
    
for md5sum in md5_collisions:
    if len(md5_collisions[md5sum]) > 1:
        for f in md5_collisions[md5sum]:
            print(f["md5sum"], f["filename"])


In [17]:
records

,source,target,md5sum,accession
2,igvf_003/nextseq/003_67A_R1.fastq.gz,igvf_003/nextseq/igvf_003_B02_D_next_N1_R1.fas...,35350d53cf858267d294bce69e475c83,https://api.data.igvf.org/sequence-files/IGVFF...
3,igvf_003/nextseq/003_67A_R2.fastq.gz,igvf_003/nextseq/igvf_003_B02_D_next_N1_R2.fas...,00c68ff0f9e0217d422c57e8948d4bb4,https://api.data.igvf.org/sequence-files/IGVFF...
4,igvf_003/nextseq/003_67B_R1.fastq.gz,igvf_003/nextseq/igvf_003_B03_D_next_N1_R1.fas...,79db765143bb23e14c613c9e1339e9f7,https://api.data.igvf.org/sequence-files/IGVFF...
5,igvf_003/nextseq/003_67B_R2.fastq.gz,igvf_003/nextseq/igvf_003_B03_D_next_N1_R2.fas...,c4787058f6b8304b1ddae0a55b48fb13,https://api.data.igvf.org/sequence-files/IGVFF...
6,igvf_003/nextseq/003_8A_R1.fastq.gz,igvf_003/nextseq/igvf_003_B01_D_next_N1_R1.fas...,bb696e63b6a8710ec8435b60ef6c81f8,
...,...,...,...,...
3098,igvf_b01/next2/B01_13F_R2.fastq.gz,igvf_b01/next2/igvf_b01_B06_D_next_N2_R2.fastq.gz,5f49893f54aaf88d06ca5834268a4e55,https://api.data.igvf.org/sequence-files/IGVFF...
3100,igvf_b01/next2/B01_13G_R1.fastq.gz,igvf_b01/next2/igvf_b01_B07_E_next_N2_R1.fastq.gz,2d4a0beb2e31d5981ebc56748297395b,https://api.data.igvf.org/sequence-files/IGVFF...
3102,igvf_b01/next2/B01_13G_R2.fastq.gz,igvf_b01/next2/igvf_b01_B07_E_next_N2_R2.fastq.gz,b3f5e2b91fc926c070c85c40e654cb0d,https://api.data.igvf.org/sequence-files/IGVFF...
3104,igvf_b01/next2/B01_13H_R1.fastq.gz,igvf_b01/next2/igvf_b01_B08_D_next_N2_R1.fastq.gz,cef06002f2986f0c305da0031910c1f0,https://api.data.igvf.org/sequence-files/IGVFF...
